# Démo 2 — Paramètres de génération

**Contexte :** assistant du service client de Lagune Banque.

Cette démo compare les effets de `temperature`, `top_p` et `seed` sur les réponses, puis montre comment `max_tokens` limite leur longueur.

**Sommaire**
- [Configuration](#configuration)
- [Variabilité des réponses](#variabilite)
- [Longueur maximale](#longueur-maximale)
- [À retenir](#a-retenir)

## <a id="configuration"></a>Configuration

In [1]:
import json
import os
import re
import sys
import time
from pathlib import Path

import litellm
import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../../../code_partage")] if (p / "config.py").exists()).resolve()))
from config import prepare

In [ ]:
prepare()

In [2]:
SEED = 42


def ask(messages, **options):
    debut = time.perf_counter()
    r = litellm.completion(model=options.pop("model", os.environ["MODELE"]), api_key=os.environ["LLM_API_KEY"], seed=SEED, num_retries=4, messages=messages, **options)
    return {"text": (r.choices[0].message.content or "").strip(), "in": r.usage.prompt_tokens, "out": r.usage.completion_tokens, "seconds": time.perf_counter() - debut, "raw": r}

## <a id="variabilite"></a>Variabilité des réponses

Le même prompt est envoyé huit fois. Le tableau compare le nombre de réponses distinctes pour `temperature=0`, `temperature=1.5` et un `seed` fixe sans température.

In [ ]:
N = 8
MODELES = [m for m in (os.getenv("MODELE"), os.getenv("MODELE_2")) if m]
PROMPT = [{"role": "user", "content": "Écris une phrase originale et inattendue de 12 mots sur un éléphant bancaire."}]


def distinct(modele, **options):
    reponses = []
    for _ in range(N):
        r = litellm.completion(model=modele, api_key=os.environ["LLM_API_KEY"], messages=PROMPT, max_tokens=400, num_retries=4, **options)
        reponses.append((r.choices[0].message.content or "").strip())
    return len(set(reponses))


COMPTES = {modele: {nom: distinct(modele, **options) for nom, options in {"temperature = 0": {"temperature": 0}, "temperature = 1,5": {"temperature": 1.5}, "seed fixé, sans température": {"seed": 1}}.items()} for modele in MODELES}
pd.DataFrame({modele: {nom: f"{n} / {N}" for nom, n in comptes.items()} for modele, comptes in COMPTES.items()}).rename_axis("réglage")

In [4]:
for modele, comptes in COMPTES.items():
    froid, chaud, graine = comptes["temperature = 0"], comptes["temperature = 1,5"], comptes["seed fixé, sans température"]
    effet = "temperature=0 rend le modèle déterministe : le réglage agit" if froid <= 2 else "temperature=0 ne rend PAS le modèle déterministe : le réglage est ignoré"
    print(f"- {modele} : {effet} (0 : {froid}/{N} réponses distinctes, 1,5 : {chaud}/{N}, seed fixé : {graine}/{N}).")

- gemini/gemini-3.5-flash-lite : temperature=0 ne rend PAS le modèle déterministe : le réglage est ignoré (0 : 8/8 réponses distinctes, 1,5 : 8/8, seed fixé : 1/8).
- gemini/gemini-3.1-flash-lite : temperature=0 rend le modèle déterministe : le réglage agit (0 : 1/8 réponses distinctes, 1,5 : 8/8, seed fixé : 1/8).


Les résultats décrivent le comportement du modèle testé ; ils ne constituent pas une garantie pour d'autres modèles ou d'autres appels. Un `seed` fixe peut améliorer la reproductibilité, mais son effet doit être vérifié. Une consigne précise et le mode structuré (démo 5) sont d'autres moyens de guider la sortie.

## <a id="longueur-maximale"></a>Longueur maximale

`max_tokens` fixe un plafond à la réponse. Si le plafond est atteint, la réponse peut être tronquée ; l'API l'indique avec `finish_reason = "length"`.

In [5]:
for limite in (20, 400):
    r = ask([{"role": "user", "content": "Explique en quelques phrases ce qu'est un prêt à la consommation."}], max_tokens=limite)["raw"]
    print(f"max_tokens={limite:3d} | finish_reason={r.choices[0].finish_reason!r:8} | {r.usage.completion_tokens} tokens | {(r.choices[0].message.content or '')[:80]!r}…")

max_tokens= 20 | finish_reason='length' | 16 tokens | 'Un **prêt à la consommation** est un crédit accordé par un organisme'…
max_tokens=400 | finish_reason='stop'   | 109 tokens | 'Un **prêt à la consommation** est un crédit accordé par un organisme financier à'…


## <a id="a-retenir"></a>À retenir

- Le comportement des paramètres dépend du modèle et doit être vérifié.
- `temperature` et `top_p` influencent l'échantillonnage ; certains modèles limitent ces réglages.
- Un `seed` fixe peut rendre les essais plus reproductibles, sans garantie universelle.
- `max_tokens` limite la longueur et `finish_reason = "length"` signale une réponse coupée.